In [1]:
import pandas as pd
import io

def vcf_to_pandas_direct_top100(file_path, nrows=100):
    lines = []
    
    with open(file_path, 'r') as f:
        # 1. Quét qua các dòng metadata cho đến khi gặp dòng tiêu đề
        for line in f:
            if line.startswith('##'):
                continue
            if line.startswith('#CHROM'):
                lines.append(line) # Lưu lại dòng tiêu đề (header)
                break              # Dừng quét metadata
                
        # 2. Đọc tiếp đúng số dòng dữ liệu được chỉ định (nrows)
        for i, line in enumerate(f):
            if i >= nrows:
                break
            lines.append(line)
    
    # 3. Đưa list 101 chuỗi (1 header + 100 data) vào pandas
    df = pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str}, # Đảm bảo nhiễm sắc thể (như 'X', 'Y') không bị lỗi kiểu dữ liệu
        sep='\t'
    )
    
    # Đổi tên cột '#CHROM' thành 'CHROM' cho dễ thao tác
    df = df.rename(columns={'#CHROM': 'CHROM'})
    return df

# Cách gọi hàm
df = vcf_to_pandas_direct_top100(r"C:\Users\dotru\STUDIE\FPTU\AiTA_Lab\data\ver3\dbVar_GRCh38.variant_call.clinical.pathogenic_or_likely_pathogenic.vcf")
df.to_csv(r"C:\Users\dotru\STUDIE\FPTU\AiTA_Lab\data\ver3\dbVar_patho_sample.tsv", sep = '\t', index=False)
df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,1,10001,nssv16255736,T,<DEL>,.,.,DBVARID=nssv16255736;SVTYPE=DEL;IMPRECISE;END=...
1,1,10001,nssv18792863,T,<DEL>,.,.,DBVARID=nssv18792863;SVTYPE=DEL;IMPRECISE;END=...
2,1,14874,nssv15124731,G,<DEL>,.,.,DBVARID=nssv15124731;SVTYPE=DEL;IMPRECISE;END=...
3,1,14874,nssv15127013,G,<DEL>,.,.,DBVARID=nssv15127013;SVTYPE=DEL;IMPRECISE;END=...
4,1,19225,nssv15150044,C,<DUP>,.,.,DBVARID=nssv15150044;SVTYPE=DUP;IMPRECISE;END=...
...,...,...,...,...,...,...,...,...
95,1,914086,nssv15151211,A,<DEL>,.,.,DBVARID=nssv15151211;SVTYPE=DEL;IMPRECISE;END=...
96,1,914086,nssv15151888,A,<DEL>,.,.,DBVARID=nssv15151888;SVTYPE=DEL;IMPRECISE;END=...
97,1,914086,nssv15154020,A,<DEL>,.,.,DBVARID=nssv15154020;SVTYPE=DEL;IMPRECISE;END=...
98,1,914086,nssv15156370,A,<DEL>,.,.,DBVARID=nssv15156370;SVTYPE=DEL;IMPRECISE;END=...


In [7]:
import pandas as pd
import re

def extract_clinical_info_from_vcf(file_path):
    records = []
    
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('#'): continue
                
            cols = line.strip().split('\t')
            if len(cols) < 8: continue
                
            chrom = cols[0]
            pos = cols[1]
            ref = cols[3]
            alt = cols[4]
            sv_id = cols[2]
            info = cols[7]
            
            # --- Dùng Regex để tìm các thẻ thông tin lâm sàng trong cột INFO ---
            
            # Tìm thẻ CLNSIG (Ý nghĩa lâm sàng)
            clnsig_match = re.search(r'CLNSIG=([^;]+)', info)
            clnsig = clnsig_match.group(1) if clnsig_match else None
            
            # Tìm thẻ CLNACC (Mã ClinVar)
            clnacc_match = re.search(r'CLNACC=([^;]+)', info)
            clnacc = clnacc_match.group(1) if clnacc_match else None
            
            # Tìm thẻ PHENO (Mã bệnh lý OMIM/MedGen)
            pheno_match = re.search(r'PHENO=([^;]+)', info)
            pheno = pheno_match.group(1) if pheno_match else None

            # Tìm thẻ PHENO (Mã bệnh lý OMIM/MedGen)
            type_match = re.search(r'SVTYPE=([^;]+)', info)
            type = type_match.group(1) if type_match else None
            
            # Lưu lại vào danh sách
            records.append({
                'CHROM': chrom,
                'POS': pos,
                'REF': ref,
                'ALT': alt,
                'ID': sv_id,
                'SVTYPE': type,
                'CLNSIG': clnsig,
                'CLINVAR_ID': clnacc,
                'DISEASE_ID': pheno
            })
                
    return pd.DataFrame(records)

# Chạy thử
df_clinical = extract_clinical_info_from_vcf(r"C:\Users\dotru\STUDIE\FPTU\AiTA_Lab\data\ver3\dbVar_GRCh38.variant_call.clinical.pathogenic_or_likely_pathogenic.vcf")
df_clinical

,CHROM,POS,REF,ALT,ID,SVTYPE,CLNSIG,CLINVAR_ID,DISEASE_ID
0,1,10001,T,<DEL>,nssv16255736,DEL,"""Pathogenic""","RCV001260116.1,VCV000980940.1","MedGen:C3661900,OMIM:104250.0001,OMIM:109630.0001"
1,1,10001,T,<DEL>,nssv18792863,DEL,"""Pathogenic""","RCV003226604.2,VCV002501007.2","MONDO:0011929,MeSH:C535362,MedGen:C1842870,OMI..."
2,1,14874,G,<DEL>,nssv15124731,DEL,"""Pathogenic""","RCV000450793.3,VCV000398920.2","""See%20cases"""
3,1,14874,G,<DEL>,nssv15127013,DEL,"""Pathogenic""","RCV000451919.3,VCV000399104.2","""See%20cases"""
4,1,19225,C,<DUP>,nssv15150044,DUP,"""Pathogenic""","RCV000447000.5,VCV000393899.4","""See%20cases"""
...,...,...,...,...,...,...,...,...,...
33699,Y,22674569,C,<DEL>,nssv18846493,DEL,"""Pathogenic""","RCV004442848.2,VCV003148951.2","MedGen:C3661900,OMIM:104250.0001,OMIM:109630.0001"
33700,Y,22727003,G,<DEL>,nssv15135476,DEL,"""Pathogenic""","RCV000138341.5,VCV000149291.2","""See%20cases"""
33701,Y,23991152,A,<DEL>,nssv15150735,DEL,"""Likely%20pathogenic""","RCV000512115.4,VCV000442918.3","""See%20cases"""
33702,Y,24095152,A,<DEL>,nssv16208654,DEL,"""Pathogenic""","RCV001007401.1,VCV000816447.1",MedGen:CN517202


In [8]:
df_clinical.dropna(subset=['CLNSIG'])

,CHROM,POS,REF,ALT,ID,SVTYPE,CLNSIG,CLINVAR_ID,DISEASE_ID
0,1,10001,T,<DEL>,nssv16255736,DEL,"""Pathogenic""","RCV001260116.1,VCV000980940.1","MedGen:C3661900,OMIM:104250.0001,OMIM:109630.0001"
1,1,10001,T,<DEL>,nssv18792863,DEL,"""Pathogenic""","RCV003226604.2,VCV002501007.2","MONDO:0011929,MeSH:C535362,MedGen:C1842870,OMI..."
2,1,14874,G,<DEL>,nssv15124731,DEL,"""Pathogenic""","RCV000450793.3,VCV000398920.2","""See%20cases"""
3,1,14874,G,<DEL>,nssv15127013,DEL,"""Pathogenic""","RCV000451919.3,VCV000399104.2","""See%20cases"""
4,1,19225,C,<DUP>,nssv15150044,DUP,"""Pathogenic""","RCV000447000.5,VCV000393899.4","""See%20cases"""
...,...,...,...,...,...,...,...,...,...
33699,Y,22674569,C,<DEL>,nssv18846493,DEL,"""Pathogenic""","RCV004442848.2,VCV003148951.2","MedGen:C3661900,OMIM:104250.0001,OMIM:109630.0001"
33700,Y,22727003,G,<DEL>,nssv15135476,DEL,"""Pathogenic""","RCV000138341.5,VCV000149291.2","""See%20cases"""
33701,Y,23991152,A,<DEL>,nssv15150735,DEL,"""Likely%20pathogenic""","RCV000512115.4,VCV000442918.3","""See%20cases"""
33702,Y,24095152,A,<DEL>,nssv16208654,DEL,"""Pathogenic""","RCV001007401.1,VCV000816447.1",MedGen:CN517202


In [9]:
df_clinical['SVTYPE'].unique()

array(['DEL', 'DUP', 'INV', 'INS', 'BND'], dtype=object)